In [1]:
#https://stackoverflow.com/questions/74071638/create-bookmarks-for-separate-pdf-files-using-the-filename
!pip install PyPDF2
!pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 31.0 MB/s eta 0:00:00


## imports

In [2]:
import os
from os import listdir
from os.path import isfile, join
from PyPDF2 import PdfFileMerger, PdfFileReader, PdfFileWriter, PdfReader, PdfWriter , PdfMerger
from google.colab import files
import datetime
from datetime import datetime
import shutil
from PyPDF2.generic import RectangleObject


## Getting Timestamp

In [3]:
from datetime import datetime
from datetime import date

# storing the current date/time in the variable
today = date.today()
c = datetime.now()

# Displays Date/Time
time_stamp = "%02d%02d.%s" % (today.month, today.day, c.strftime('%2H%2M%2S'))
print("time_stamp = ", time_stamp)

time_stamp =  1101.082948


## upload_files_2_dir: Upload files, and move those files to a specified directory

In [4]:
def upload_files_2_dir(orig_file_dir=None):
  from datetime import datetime
  from datetime import date
  from google.colab import files
  from tqdm.notebook import tqdm

  if orig_file_dir is None or orig_file_dir == "":
    # storing the current date/time in the variable
    today = date.today()
    c = datetime.now()
    # Displays Date/Time
    time_stamp = "%02d%02d.%s" % (today.month, today.day, c.strftime('%2H%2M%2S'))
    #print("time_stamp = ", time_stamp)
    folder_orgi_files = f'orig_{time_stamp}'
  else:
    folder_orgi_files = orig_file_dir

  print(f'folder of files to be processed: {folder_orgi_files}')
  uploaded = files.upload()

  try:
    #folder_orgi_files = f'orig_{time_stamp}'
    os.makedirs(f'{folder_orgi_files}', exist_ok=True)

    for f in tqdm(uploaded.keys(), desc="Uploading"):
      print('User uploaded file "{name}" with length {length} bytes, moving to {orig_folder}'.\
          format(name=f, length=len(uploaded[f]), orig_folder=f'{folder_orgi_files}/{f}'))
      shutil.move(f, f"{folder_orgi_files}/{f}")
  except Exception as e:
    print(f"Error processing: upload_files_2_dir: {e}")

  return folder_orgi_files

## add_blank_pages: add blank pages to the pdf file, and move the file to a specified folder

In [5]:
# prompt: adding blank page(s) to the end of pdf files, so the total pages are xN (N=2,4,8 etc)
def add_blank_pages(pdf_path, num_blanks, output_path):
  """Adds blank pages to the end of a PDF file.

  Args:
    pdf_path (str): Path to the input PDF file.
    num_blanks (int): The number of blank pages to add.
    output_path (str): Path to save the modified PDF file.
  """

  from reportlab.pdfgen import canvas
  from reportlab.lib.pagesizes import letter
  import io

  print(f"add_blank_pages(\n\t{pdf_path} \n\t + {num_blanks} pgs ==> \n\t {output_path})")
  try:
    reader = PdfReader(pdf_path)
    writer = PdfWriter()

    # Get the page size of the pdf file
    pg1st = reader.pages[0]
    pgMediaBox = pg1st.mediabox
    pgWidth = pgMediaBox.width
    pgHeight = pgMediaBox.height
    print(f'done 4. - pgs={len(reader.pages)}, (W.H)=({pgWidth}, {pgHeight})')

    # Define page size (e.g., A4 letter size)
    # letter size is 612 x 792 points
    pgSize = RectangleObject((0, 0, pgWidth, pgHeight))
    #pgSize = RectangleObject((0, 0, 612, 792))

    # Add original pages
    for page in range(len(reader.pages)):
      writer.add_page(reader.pages[page])

    ## Create a blank PDF page using ReportLab
    #packet = io.BytesIO()
    #can = canvas.Canvas(packet, pagesize=letter)
    #can.save() # Save the canvas to the packet
    #packet.seek(0)

    #blank_page_reader = PdfReader(packet)
    #print(f'done 7, len(blank_page_reader.pages)={len(blank_page_reader.pages)}')
    #blank_page = blank_page_reader.pages[0]

    # Add blank pages to the writer
    for _ in range(num_blanks):
        writer.add_blank_page(width=pgSize.right, height=pgSize.top)
        #writer.add_page(blank_page)

    # Write the output PDF
    with open(output_path, 'wb') as outfile:
        writer.write(outfile)

  except Exception as e:
    print(f"Error processing {pdf_path}: {e}")

## process the uploaded files from a specified folder

In [6]:
# Number of blank pages to add to maek the pdf file multiple of N pages
num_blank_xN_page = 4 #@param {type:"integer"}

In [7]:
def pdf_add_blank_pages_Nx(input_path=None, pg_Nx=num_blank_xN_page, output_path=None):
  from datetime import datetime
  from datetime import date
  from google.colab import files
  from tqdm.notebook import tqdm

  if input_path is None or input_path == "":
    folder_toBeProcessed = upload_files_2_dir()
  else:
    folder_toBeProcessed = input_path

  if output_path is None or output_path == "":
    # storing the current date/time in the variable
    today = date.today()
    c = datetime.now()
    # Displays Date/Time
    time_stamp = "%02d%02d.%s" % (today.month, today.day, c.strftime('%2H%2M%2S'))
    print("time_stamp = ", time_stamp)
    # Create a directory to save the processed files
    processed_folder = f'{os.getcwd()}/processed_{time_stamp}'
  else:
    processed_folder = output_path

  os.makedirs(processed_folder, exist_ok=True)
  print(f'processed_folder = {processed_folder}')

  #directory = f'{os.getcwd()}/{folder_toBeProcessed}'
  #source_dir = f'{os.getcwd()}/{folder_toBeProcessed}'

  #source_dir = f'{os.getcwd()}/{folder_toBeProcessed}'
  #directory = source_dir

  pdfFileList = [f for f in listdir(folder_toBeProcessed) if (isfile(join(folder_toBeProcessed, f)) and f.endswith('.pdf'))]
  pdfFileList.sort()

  pathname, extension = os.path.splitext(folder_toBeProcessed)
  filename = pathname.split('\\')

  print('source_dir =', folder_toBeProcessed)
  print('filenames = ', pdfFileList)

  # Process each PDF file
  for pdf_file in tqdm(pdfFileList, desc="Adding blank pages"):
    pdf_reader = PdfReader(f"{folder_toBeProcessed}/{pdf_file}")
    print(f"Processing file: {folder_toBeProcessed}/{pdf_file} ==> {processed_folder}/{os.path.splitext(pdf_file)[0]}.pdf")
    print(f"\t\t\t pages={len(pdf_reader.pages)}, [w,h]=[{pdf_reader.pages[0].mediabox.width},{pdf_reader.pages[0].mediabox.height}]")

    pages_to_add = ((len(pdf_reader.pages)+3) // num_blank_xN_page) * num_blank_xN_page - len(pdf_reader.pages)
    print(f"\t\t\t pages_to_add = {pages_to_add}")
    input_path = os.path.join(folder_toBeProcessed, pdf_file)
    output_path = os.path.join(processed_folder, pdf_file)
    add_blank_pages(input_path, pages_to_add, output_path)

    print(f"Finished adding {pages_to_add} blank page(s) to PDF files in '{folder_toBeProcessed}'. Modified files are in '{processed_folder}'.")

  return processed_folder

## Main processing - uploading files, adding blank pages

In [8]:
folder_toBeProcessed = upload_files_2_dir()
processed_folder = pdf_add_blank_pages_Nx(folder_toBeProcessed, 4)

folder of files to be processed: orig_1101.082948


Saving lec00-Intro.pdf to lec00-Intro.pdf
Saving lec01-DNN_Comp.pdf to lec01-DNN_Comp.pdf
Saving lec02-ML_Hardware.pdf to lec02-ML_Hardware.pdf
Saving lec03-uArch.pdf to lec03-uArch.pdf
Saving lec04-TinyML.pdf to lec04-TinyML.pdf
Saving lec05-Quantization.pdf to lec05-Quantization.pdf
Saving lec07-Knowledge Distillation.pdf to lec07-Knowledge Distillation.pdf
Saving lec08-Neural Architecture Search.pdf to lec08-Neural Architecture Search.pdf
Saving lec09-Kernel Comp, Approx..pdf to lec09-Kernel Comp, Approx..pdf
Saving lec10-Mapping.pdf to lec10-Mapping.pdf
Saving lec11-Pre_Post_Processing.pptx.pdf to lec11-Pre_Post_Processing.pptx.pdf
Saving lec12-Distributed Training.pptx.pdf to lec12-Distributed Training.pptx.pdf
Saving lec13-Federated Learning.pptx.pdf to lec13-Federated Learning.pptx.pdf
Saving lec14-Environ_Ethical_Systems_Issues.pptx.pdf to lec14-Environ_Ethical_Systems_Issues.pptx.pdf


Uploading:   0%|          | 0/14 [00:00<?, ?it/s]

User uploaded file "lec00-Intro.pdf" with length 7052744 bytes, moving to orig_1101.082948/lec00-Intro.pdf
User uploaded file "lec01-DNN_Comp.pdf" with length 2633128 bytes, moving to orig_1101.082948/lec01-DNN_Comp.pdf
User uploaded file "lec02-ML_Hardware.pdf" with length 12380145 bytes, moving to orig_1101.082948/lec02-ML_Hardware.pdf
User uploaded file "lec03-uArch.pdf" with length 2275675 bytes, moving to orig_1101.082948/lec03-uArch.pdf
User uploaded file "lec04-TinyML.pdf" with length 16456019 bytes, moving to orig_1101.082948/lec04-TinyML.pdf
User uploaded file "lec05-Quantization.pdf" with length 2629860 bytes, moving to orig_1101.082948/lec05-Quantization.pdf
User uploaded file "lec07-Knowledge Distillation.pdf" with length 917496 bytes, moving to orig_1101.082948/lec07-Knowledge Distillation.pdf
User uploaded file "lec08-Neural Architecture Search.pdf" with length 3226626 bytes, moving to orig_1101.082948/lec08-Neural Architecture Search.pdf
User uploaded file "lec09-Kernel 

Adding blank pages:   0%|          | 0/14 [00:00<?, ?it/s]

Processing file: orig_1101.082948/lec00-Intro.pdf ==> /content/processed_1101.083342/lec00-Intro.pdf
			 pages=39, [w,h]=[720,405]
			 pages_to_add = 1
add_blank_pages(
	orig_1101.082948/lec00-Intro.pdf 
	 + 1 pgs ==> 
	 /content/processed_1101.083342/lec00-Intro.pdf)
done 4. - pgs=39, (W.H)=(720, 405)
Finished adding 1 blank page(s) to PDF files in 'orig_1101.082948'. Modified files are in '/content/processed_1101.083342'.
Processing file: orig_1101.082948/lec01-DNN_Comp.pdf ==> /content/processed_1101.083342/lec01-DNN_Comp.pdf
			 pages=63, [w,h]=[720,405]
			 pages_to_add = 1
add_blank_pages(
	orig_1101.082948/lec01-DNN_Comp.pdf 
	 + 1 pgs ==> 
	 /content/processed_1101.083342/lec01-DNN_Comp.pdf)
done 4. - pgs=63, (W.H)=(720, 405)
Finished adding 1 blank page(s) to PDF files in 'orig_1101.082948'. Modified files are in '/content/processed_1101.083342'.
Processing file: orig_1101.082948/lec02-ML_Hardware.pdf ==> /content/processed_1101.083342/lec02-ML_Hardware.pdf
			 pages=114, [w,h

## Zip the processed folder

In [9]:
import shutil

zip_filename = f'pg{num_blank_xN_page}X_{time_stamp}'
shutil.make_archive(zip_filename, 'zip', processed_folder)

'/content/pg4X_1101.082948.zip'

#download file to local

In [10]:
from google.colab import files
files.download(f'{zip_filename}.zip')
print(f"Downloading {zip_filename}.zip  -- done!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>